---
  title: "Aula 3"
  author: Mauro Alixandrini
  date: 2026-16-09
  categories: [GeoAI, technology]
  image: ambiente.jpg
---

[![](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Alixandrini/PPEC083/blob/main/Notebooks/Aula_3.ipynb)

# Download de Dados de Sensoriamento Remoto
### 4.4.4 Bibliotecas Python para STAC

Para começar, conecte-se à API STAC do Planetary Computer e explore suas coleções:

In [1]:
from pystac_client import Client
catalog = Client.open("https://planetarycomputer.microsoft.com/api/stac/v1")
print(f"Catalog title: {catalog.title}")
print(f"Catalog description: {catalog.description}")
collections = list(catalog.get_collections())
print(f"\nNumber of collections: {len(collections)}")
print("\nFirst 10 collections:")
for collection in collections[:10]:
    print(f"  {collection.id}: {collection.title}")

Catalog title: Microsoft Planetary Computer STAC API
Catalog description: Searchable spatiotemporal metadata describing Earth science datasets hosted by the Microsoft Planetary Computer

Number of collections: 136

First 10 collections:
  daymet-annual-pr: Daymet Annual Puerto Rico
  daymet-daily-hi: Daymet Daily Hawaii
  3dep-seamless: USGS 3DEP Seamless DEMs
  3dep-lidar-dsm: USGS 3DEP Lidar Digital Surface Model
  fia: Forest Inventory and Analysis
  gridmet: gridMET
  daymet-annual-na: Daymet Annual North America
  daymet-monthly-na: Daymet Monthly North America
  daymet-annual-hi: Daymet Annual Hawaii
  daymet-monthly-hi: Daymet Monthly Hawaii


4.4.5.  Explorando uma Coleção STAC

Antes de pesquisar itens, é útil inspecionar os metadados de uma coleção para entender o que ela contém, sua extensão espacial e temporal, e quais propriedades estão disponíveis para filtragem.

In [2]:
collection = catalog.get_collection("sentinel-2-l2a")
print(f"Title: {collection.title}")
print(f"Description: {collection.description[:200]}...")
print(f"License: {collection.license}")
print(f"Temporal extent: {collection.extent.temporal.intervals}")
print(f"Spatial extent: {collection.extent.spatial.bboxes}")

Title: Sentinel-2 Level-2A
Description: The [Sentinel-2](https://sentinel.esa.int/web/sentinel/missions/sentinel-2) program provides global imagery in thirteen spectral bands at 10m-60m resolution and a revisit time of approximately five da...
License: proprietary
Temporal extent: [[datetime.datetime(2015, 6, 27, 10, 25, 31, tzinfo=tzutc()), None]]
Spatial extent: [[-180, -90, 180, 90]]


### 4.4.6.  Pesquisando Itens
Uma vez que você sabe quais coleções estão disponíveis, pode pesquisar itens que correspondam aos seus critérios espaciais, temporais e de qualidade. O exemplo a seguir pesquisa cenas Sentinel-2 sobre Knoxville, Tennessee, durante o verão de 2025 com baixa cobertura de nuvens:

In [3]:
bbox = [-83.95, 35.94, -83.91, 35.98]  # Small area near Knoxville, TN
search = catalog.search(
    collections=["sentinel-2-l2a"],
    bbox=bbox,
    datetime="2025-06-01/2025-08-31",
    query={"eo:cloud_cover": {"lt": 10}},
    max_items=3,
)
items = search.item_collection()
print(f"Found {len(items)} items\n")
for item in items:
    cloud_cover = item.properties.get("eo:cloud_cover")
    cloud_cover_text = (
        f"{cloud_cover:.1f}%" if cloud_cover is not None else "N/A"
    )
    print(f"ID: {item.id}")
    print(f"  Date: {item.datetime}")
    print(f"  Cloud cover: {cloud_cover_text}")
    print(f"  Bounding box: {item.bbox}")
    print()

Found 3 items

ID: S2B_MSIL2A_20250816T161829_R040_T16SGE_20250816T201128
  Date: 2025-08-16 16:18:29.024000+00:00
  Cloud cover: 9.7%
  Bounding box: [-84.8052687, 35.1072182, -83.5595813, 36.1242843]

ID: S2A_MSIL2A_20250704T162711_R040_T16SGE_20250705T010417
  Date: 2025-07-04 16:27:11.024000+00:00
  Cloud cover: 8.0%
  Bounding box: [-84.8052687, 35.1072182, -83.5595813, 36.1242843]

ID: S2C_MSIL2A_20250622T161851_R040_T16SGE_20250622T215416
  Date: 2025-06-22 16:18:51.025000+00:00
  Cloud cover: 9.9%
  Bounding box: [-84.8052687, 35.1072182, -83.5595813, 36.1242843]



### 4.4.7.  Inspecionando Itens e Ativos
Uma vez que você tem resultados de busca, inspecione um item individual para ver suas propriedades
completas e os ativos disponíveis para download. Cada item Sentinel-2 tipicamente contém ativos para
bandas espectrais individuais (B01 a B12, mais B8A), um composto visual, um mapa de classifi cação de
cena e vários arquivos de metadados.

In [5]:
if items:
    item = items[0]
    cloud_cover = item.properties.get("eo:cloud_cover")
    cloud_cover_text = (
        f"{cloud_cover}%" if cloud_cover is not None else "N/A"
    )
    print(f"Item ID: {item.id}")
    print(f"Date: {item.datetime}")
    print(f"Cloud cover: {cloud_cover_text}")
    print(f"Platform: {item.properties.get('platform', 'N/A')}")
    print(f"\nAvailable assets ({len(item.assets)}):")
    for key, asset in item.assets.items():
        roles = ", ".join(asset.roles) if asset.roles else "N/A"
        print(f"  {key}: {asset.title or 'No title'} [{roles}]")

Item ID: S2B_MSIL2A_20250816T161829_R040_T16SGE_20250816T201128
Date: 2025-08-16 16:18:29.024000+00:00
Cloud cover: 9.708405%
Platform: Sentinel-2B

Available assets (23):
  AOT: Aerosol optical thickness (AOT) [data]
  B01: Band 1 - Coastal aerosol - 60m [data]
  B02: Band 2 - Blue - 10m [data]
  B03: Band 3 - Green - 10m [data]
  B04: Band 4 - Red - 10m [data]
  B05: Band 5 - Vegetation red edge 1 - 20m [data]
  B06: Band 6 - Vegetation red edge 2 - 20m [data]
  B07: Band 7 - Vegetation red edge 3 - 20m [data]
  B08: Band 8 - NIR - 10m [data]
  B09: Band 9 - Water vapor - 60m [data]
  B11: Band 11 - SWIR (1.6) - 20m [data]
  B12: Band 12 - SWIR (2.2) - 20m [data]
  B8A: Band 8A - Vegetation red edge 4 - 20m [data]
  SCL: Scene classfication map (SCL) [data]
  WVP: Water vapour (WVP) [data]
  visual: True color image [data]
  safe-manifest: SAFE manifest [metadata]
  granule-metadata: Granule metadata [metadata]
  inspire-metadata: INSPIRE metadata [metadata]
  product-metadata: Produ

## 4.5.  Download de Imagens NAIP
O pacote  geoai  fornece uma função conveniente  download_naip()  que encapsula o fluxo de trabalho de busca e download STAC em uma única chamada. Ela pesquisa o Planetary Computer por imagens NAIP que intersectam sua caixa delimitadora, baixa os resultados como arquivos GeoTIFF e retorna uma lista de caminhos de arquivos.

In [6]:
import geoai
bbox = [-83.94, 35.96, -83.92, 35.98]  # Small area near Knoxville, TN
output_dir = "naip_data"
filepaths = geoai.download_naip(
    bbox=bbox,
    output_dir=output_dir,
    year=2021,
    max_items=1,
)
print(f"Downloaded {len(filepaths)} file(s):")
for fp in filepaths:
    print(f"  {fp}")

c:\Users\mauro\.conda\envs\geoai\Lib\site-packages\pyproj\network.py:59: UserWarning: pyproj unable to set PROJ database path.
  _set_context_ca_bundle_path(ca_bundle_path)


m_3508301_nw_17_060_20210403.tif:   0%|          | 0.00/451M [00:00<?, ?iB/s]

Downloaded 1 file(s):
  naip_data\m_3508301_nw_17_060_20210403.tif


Os parâmetros da função são:
* **bbox**: Uma tupla de (min_lon, min_lat, max_lon, max_lat) em coordenadas WGS84
* **output_dir**: Diretório onde os arquivos GeoTIFF baixados são salvos
* **year**: Ano de aquisição NAIP específi co (por exemplo, 2021). Se  None , retorna imagens de todos os
anos disponíveis
* **max_items**: Número máximo de tiles de cena para baixar
Após o download, você pode inspecionar as imagens com  rasterio  para verifi car suas propriedades:

In [3]:
import rasterio
if filepaths:
    with rasterio.open(filepaths[0]) as src:
        print(f"Dimensions: {src.width} x {src.height}")
        print(f"Bands: {src.count}")
        print(f"CRS: {src.crs}")
        print(f"Resolution: {src.res[0]:.2f} m")
        print(f"Bounds: {src.bounds}")
        print(f"Data type: {src.dtypes[0]}")

NameError: name 'filepaths' is not defined

As imagens NAIP são entregues como GeoTIFFs de 4 bandas com bandas ordenadas como Vermelho, Verde, Azul e Infravermelho Próximo (RGBN). O tipo de dado típico é  uint8  com valores variando de 0 a 255.


Alternativamente, você pode usar a biblioteca  geoai  para imprimir metadados raster em uma única linha
de código:

In [6]:
geoai.print_raster_info(filepaths[0])

===== RASTER INFORMATION: naip_data\m_3508301_nw_17_060_20210403.tif =====
Driver: GTiff
Dimensions: 10420 x 12520 pixels
Number of bands: 4
Data type: uint8
Coordinate Reference System: EPSG:26917
Georeferenced Bounds: BoundingBox(left=229164.0, bottom=3980802.0, right=235416.0, top=3988314.0)
Pixel Resolution: 0.6, 0.6
NoData Value: 0.0
----- Band Statistics -----
Band 1:
  Min: 10.00
  Max: 232.00
  Mean: 114.93
  Std Dev: 46.55
Band 2:
  Min: 20.00
  Max: 229.00
  Mean: 123.00
  Std Dev: 38.65
Band 3:
  Min: 28.00
  Max: 228.00
  Mean: 101.23
  Std Dev: 42.91
Band 4:
  Min: 1.00
  Max: 241.00
  Mean: 156.04
  Std Dev: 52.85


: 

In [5]:
# para solucionar problemas de DLLs no Windows, especialmente com GDAL e PROJ, é importante configurar corretamente as variáveis de ambiente e garantir que o diretório bin do Conda seja priorizado na busca por DLLs. O código acima faz exatamente isso, garantindo que a execução do script seja mais estável e evitando conflitos entre diferentes versões de bibliotecas.
import os
import sys

# 1. Impede crash por múltiplas instâncias da runtime OpenMP (PyTorch + GDAL)
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

# 2. Garante isolamento estrito das DLLs e dados do Conda
conda_prefix = sys.prefix
proj_dir = os.path.join(conda_prefix, "Library", "share", "proj")
gdal_dir = os.path.join(conda_prefix, "Library", "share", "gdal")

os.environ["PROJ_DATA"] = proj_dir
os.environ["GDAL_DATA"] = gdal_dir

# 3. Força prioridade do diretório bin do Conda para busca de DLLs no Windows
if hasattr(os, "add_dll_directory"):
    dll_dir = os.path.join(conda_prefix, "Library", "bin")
    if os.path.exists(dll_dir):
        os.add_dll_directory(dll_dir)

import rasterio
import geoai
bbox = [-83.94, 35.96, -83.92, 35.98]  # Small area near Knoxville, TN
output_dir = "naip_data"
filepaths = geoai.download_naip(
    bbox=bbox,
    output_dir=output_dir,
    year=2021,
    max_items=1,
)

Além do Planetary Computer, o USDA Natural Resources Conservation Service (NRCS) hospeda imagens NAIP em uma pasta pública do Box em https://nrcs.app.box.com/v/naip. A pasta é organizada por ano e estado, facilitando a navegação e o download de imagens para regiões específi cas. Isso pode ser uma alternativa conveniente quando você deseja explorar visualmente a cobertura disponível ou baixar imagens para um estado ou condado inteiro sem escrever código.

## 4.6.  Download de Dados Sentinel-2

Para  dados  Sentinel-2,  o  fl uxo  de  trabalho  tem  duas  etapas:  primeiro  pesquisar  itens  usando pystac_client , depois baixar bandas específi cas usando  geoai.download_pc_stac_item() . Isso lhe dá controle preciso sobre quais bandas e resoluções recuperar.

### 4.6.1.  Pesquisando Itens Sentinel-2

In [2]:
from pystac_client import Client
catalog = Client.open("https://planetarycomputer.microsoft.com/api/stac/v1")
bbox = [-83.94, 35.96, -83.92, 35.98]
search = catalog.search(
    collections=["sentinel-2-l2a"],
    bbox=bbox,
    datetime="2025-06-01/2025-08-31",
    query={"eo:cloud_cover": {"lt": 10}},
    max_items=1,
)
items = search.item_collection()
if items:
    item = items[0]
    cloud_cover = item.properties.get("eo:cloud_cover")
    cloud_cover_text = (
        f"{cloud_cover}%" if cloud_cover is not None else "N/A"
    )
    print(f"Selected item: {item.id}")
    print(f"Date: {item.datetime}")
    print(f"Cloud cover: {cloud_cover_text}")
    item_url = item.self_href
    print(f"Item URL: {item_url}")

Selected item: S2B_MSIL2A_20250816T161829_R040_T16SGE_20250816T201128
Date: 2025-08-16 16:18:29.024000+00:00
Cloud cover: 9.708405%
Item URL: https://planetarycomputer.microsoft.com/api/stac/v1/collections/sentinel-2-l2a/items/S2B_MSIL2A_20250816T161829_R040_T16SGE_20250816T201128


### 4.6.2.  Download e Mesclagem de Bandas
Com a URL do item em mãos, use  geoai.download_pc_stac_item()  para baixar bandas específicas e opcionalmente mesclá-las em um único GeoTIFF multibanda:

In [5]:
if items:
    result = geoai.download_pc_stac_item(
        item_url=item_url,
        bands=["B02", "B03", "B04", "B08"],  # Blue, Green, Red, NIR
        output_dir="sentinel2_data",
        merge_bands=True,
        merged_filename="knoxville_s2_rgbn.tif",
        overwrite=False,
    )
    for band_name, path in result.items():
        print(f"  {band_name}: {path}")

c:\Users\mauro\.conda\envs\geoai\Lib\site-packages\pystac_client\item_search.py:925: FutureWarning: get_items() is deprecated, use items() instead
  warnings.warn(


  B02: sentinel2_data\S2B_MSIL2A_20250816T161829_R040_T16SGE_20250816T201128_B02.tif
  B03: sentinel2_data\S2B_MSIL2A_20250816T161829_R040_T16SGE_20250816T201128_B03.tif
  B04: sentinel2_data\S2B_MSIL2A_20250816T161829_R040_T16SGE_20250816T201128_B04.tif
  B08: sentinel2_data\S2B_MSIL2A_20250816T161829_R040_T16SGE_20250816T201128_B08.tif
  merged: sentinel2_data\knoxville_s2_rgbn.tif


O  parâmetro  merge_bands=True   reamostra  todas  as  bandas  solicitadas  para  uma  resolução  comum (padrão para a resolução da primeira banda) e as empilha em um único GeoTIFF. Isso é particularmente útil ao combinar bandas em diferentes resoluções nativas, como as bandas visíveis de 10 m com as bandas de borda vermelha de 20 m. Você pode controlar a resolução de saída explicitamente com o parâmetro cell_size .

## 4.7.  Download de Dados Landsat

Os dados Landsat seguem o mesmo padrão de busca e download. O Planetary Computer hospeda dados tanto do Landsat 8 quanto do Landsat 9 na coleção  landsat-c2-l2  (Collection 2, Level 2 reflectância de superfície).

In [6]:
from pystac_client import Client
catalog = Client.open("https://planetarycomputer.microsoft.com/api/stac/v1")
bbox = [-83.94, 35.96, -83.92, 35.98]
search = catalog.search(
    collections=["landsat-c2-l2"],
    bbox=bbox,
    datetime="2025-06-01/2025-08-31",
    query={"eo:cloud_cover": {"lt": 10}},
    max_items=1,
)
items = search.item_collection()
if items:
    item = items[0]
    cloud_cover = item.properties.get("eo:cloud_cover")
    cloud_cover_text = (
        f"{cloud_cover}%" if cloud_cover is not None else "N/A"
    )
    print(f"Selected item: {item.id}")
    print(f"Date: {item.datetime}")
    print(f"Cloud cover: {cloud_cover_text}")
    print(f"\nAvailable assets:")
    for key in list(item.assets.keys())[:10]:
        print(f"  {key}")

Selected item: LC09_L2SP_019035_20250816_02_T1
Date: 2025-08-16 16:12:03.296497+00:00
Cloud cover: 8.2%

Available assets:
  qa
  ang
  red
  blue
  drad
  emis
  emsd
  trad
  urad
  atran


Para  baixar  bandas  específicas  do  Landsat,  use  a  mesma  função  geoai.download_pc_stac_item() .
As bandas do Landsat usam convenções de nomenclatura diferentes do Sentinel-2 (por exemplo, “blue”, “green”, “red”, “nir08” para Landsat vs. “B02”, “B03”, “B04”, “B08” para Sentinel-2):

In [7]:
if items:
    landsat_url = items[0].self_href
    result = geoai.download_pc_stac_item(
        item_url=landsat_url,
        bands=["blue", "green", "red", "nir08"],
        output_dir="landsat_data",
        merge_bands=True,
        merged_filename="knoxville_landsat_rgbn.tif",
        overwrite=False,
    )
    for band_name, path in result.items():
        print(f"  {band_name}: {path}")

c:\Users\mauro\.conda\envs\geoai\Lib\site-packages\pystac_client\item_search.py:925: FutureWarning: get_items() is deprecated, use items() instead
  warnings.warn(


  blue: landsat_data\LC09_L2SP_019035_20250816_02_T1_blue.tif
  green: landsat_data\LC09_L2SP_019035_20250816_02_T1_green.tif
  red: landsat_data\LC09_L2SP_019035_20250816_02_T1_red.tif
  nir08: landsat_data\LC09_L2SP_019035_20250816_02_T1_nir08.tif
  merged: landsat_data\knoxville_landsat_rgbn.tif


Alternativamente, você pode usar  geoai.pc_stac_download()  quando já tem objetos de itens STAC e deseja baixar ativos específi cos com múltiplos workers:

In [ ]:
if items:
    downloaded = geoai.pc_stac_download(
        items=items[0],
        output_dir="landsat_data_raw",
        assets=["blue", "green", "red", "nir08"],
        max_workers=4,
    )
    for item_id, assets in downloaded.items():
        print(f"Item: {item_id}")
        for asset_key, fpath in assets.items():
            print(f"  {asset_key}: {fpath}")

## 4.8.  Download de Dados Abertos Vantor
Como introduzido anteriormente neste capítulo, o Programa de Dados Abertos da Vantor fornece imagens de satélite gratuitas pré-evento e pós-evento para grandes desastres naturais. Como os dados são publicados como um catálogo STAC, você pode pesquisar e acessá-los usando o mesmo fluxo de trabalho pystac_client  usado nas seções anteriores.


O exemplo a seguir se conecta ao catálogo STAC de Dados Abertos da Vantor e lista as coleções de eventos de desastre disponíveis:

In [ ]:
from pystac_client import Client
vantor_catalog_url = (
    "https://vantor-opendata.s3.amazonaws.com/events/catalog.json"
)
vantor_catalog = Client.open(vantor_catalog_url)
collections = list(vantor_catalog.get_collections())
print(f"Number of event collections: {len(collections)}")
for collection in collections:
    print(f"  {collection.id}: {collection.title}")

Você pode então inspecionar uma coleção de evento específica para revisar seus metadados:

In [ ]:
if collections:
    event = collections[0]
    print(f"Event: {event.title}")
    print(f"Description: {event.description}")
    print(f"License: {event.license}")
    print(f"Temporal extent: {event.extent.temporal.intervals}")
    print(f"Spatial extent: {event.extent.spatial.bboxes}")

## 4.9.  Acesso a Dados Vetoriais
Muitos fluxos de trabalho de GeoAI requerem dados vetoriais junto com imagens de satélite. Pegadas de edificações servem como rótulos de treinamento para modelos de segmentação, redes viárias definem feições de interesse e polígonos de uso do solo fornecem alvos de classificação. O pacote  geoai  fornece funções convenientes para acessar grandes datasets vetoriais.

### 4.9.1.  Edificações do Overture Maps
A Overture Maps Foundation produz dados de mapa abertos, incluindo um dataset global de pegadas de edificações derivado de múltiplas fontes como OpenStreetMap, Microsoft  ML Buildings e Google Open Buildings. O pacote  geoai  oferece duas maneiras convenientes de acessar esses dados.


A função  download_overture_buildings()  usa a CLI do Overture Maps para baixar dados de edificações para uma caixa delimitadora e salvá-los em um arquivo:

In [ ]:
import geoai
bbox = (-83.94, 35.96, -83.92, 35.98)  # Knoxville area
output_path = "buildings.geojson"
geoai.download_overture_buildings(
    bbox=bbox,
    output=output_path,
    overture_type="building",
)
print(f"Buildings saved to {output_path}")

Para  mais  flexibilidade,  get_overture_data()   retorna  dados  diretamente  como  um  GeoDataFrame, permitindo filtrar por colunas e explorar interativamente. Ela suporta múltiplos tipos de dados além de edificações, incluindo places, roads, land cover e water features:

In [ ]:
gdf = geoai.get_overture_data(
    overture_type="building",
    bbox=(-83.94, 35.96, -83.92, 35.98),
    output="buildings.parquet",
)
print(f"Downloaded {len(gdf)} buildings")
gdf.head()

As  funções  do  Overture  Maps  requerem  o  pacote  overturemaps .  Instale-o  com pip install overturemaps  se ainda não estiver disponível em seu ambiente. Os downloads podem levar alguns minutos, dependendo do tamanho da área e da velocidade da sua rede.
### 4.9.2.  Dados OpenStreetMap
O OpenStreetMap (OSM) é um mapa colaborativo e open-source do mundo mantido por uma comunidade global de voluntários. Embora o Overture Maps esteja cada vez mais popular para pegadas de edificações, o OSM continua sendo uma fonte valiosa para estradas, pontos de interesse, polígonos de uso do solo e muitas outras feições. Bibliotecas como  osmnx  e  ohsome  fornecem interfaces Python para consultar dados do OSM. Para projetos de GeoAI, os dados do OSM são comumente usados para gerar rótulos de treinamento para extração de estradas, detecção de edificações e classificação de uso do solo.

A biblioteca  quackosm  usa DuckDB para extrair dados do OSM efi cientemente de arquivos PBF, oferecendo alto desempenho para grandes datasets. O pacote  leafmap  fornece funções wrapper convenientes em seu módulo  leafmap.osm  que facilitam a consulta de dados do OSM por caixa delimitadora, nome de lugar ou geometria personalizada.


Para  baixar  pegadas  de  edificações  do  OSM  para  uma  caixa  delimitadora,  use quackosm_gdf_from_bbox()  com um filtro de tags do OSM:

In [ ]:
import leafmap.osm as osm
bbox = (-83.94, 35.96, -83.92, 35.98)  # Knoxville area
buildings = osm.quackosm_gdf_from_bbox(bbox, tags={"building": True})
print(f"Downloaded {len(buildings)} buildings")
buildings.head()

Você também pode consultar dados do OSM por nome de lugar. O exemplo a seguir baixa a rede viária de Knoxville, Tennessee:

In [ ]:
roads = osm.quackosm_gdf_from_place("Knoxville, Tennessee", tags={"highway":
True})
print(f"Downloaded {len(roads)} road segments")
roads.head()

Para controle mais preciso sobre a área de consulta, passe uma geometria Shapely ou string WKT para quackosm_gdf_from_geometry() :

In [ ]:
from shapely.geometry import Polygon
polygon = Polygon([
    (-83.94, 35.96),
    (-83.92, 35.96),
    (-83.92, 35.98),
    (-83.94, 35.98),
])
natural = osm.quackosm_gdf_from_geometry(polygon, tags={"natural": True})
print(f"Downloaded {len(natural)} natural features")
natural.head()

As funções  quackosm  aceitam qualquer tag OSM válida para fi ltragem. Tags comuns úteis para GeoAI incluem  building ,  highway ,  landuse ,  natural ,  waterway  e  amenity . O primeiro download para uma região pode demorar mais porque o  quackosm  precisa buscar e armazenar em cache o arquivo PBF regional. Consultas subsequentes na mesma região reutilizam o arquivo em cache e executam muito mais rápido.

## 4.10.  Organizando Seus Dados
O gerenciamento de dados pode parecer mundano comparado ao treinamento de redes neurais, mas tem um impacto desproporcional no sucesso do projeto. Datasets mal organizados levam a tempo desperdiçado procurando arquivos, uso acidental de dados desatualizados e resultados que não podem ser reproduzidos.

À medida que seu projeto cresce de algumas imagens de teste para centenas de cenas de satélite com rótulos e saídas de modelo correspondentes, uma estrutura de diretórios limpa previne confusão e mantém os fl uxos de trabalho reproduzíveis. Um layout recomendado para projetos de GeoAI é:
<br>

project/ <br>
├── data/ <br>
│   ├── raw/..................# Arquivos originais baixados (never modified)<br>
│   │   ├── naip/<br>
│   │   ├── sentinel2/<br>
│   │   ├── landsat/<br>
│   │   └── vectors/<br>
│   ├── processed/.....# Arquivos recortados, reprojetados ou mosaicos<br>
│   └── training/..........# Tiles and labels prontos para treinar o modelo<br>
├── models/...............# Configurações e pesos dos modelos salvos<br>
├── notebooks/........# Jupyter notebooks para explicações<br>
├── scripts/.................# Escripts Python reutilizaveis<br>
└── results/.................# Saídas com predições, mapas e avaliações<br>

Siga estas convenções de nomenclatura para manter os arquivos identificáveis:
* **Inclua a fonte**:  naip_2021_knoxville.tif ,  s2_20240615_B04.tif
* **Inclua a data**: Use o formato ISO (YYYY-MM-DD ou YYYYMMDD) para que os arquivos sejam classificados cronologicamente.
* **Inclua a área**: Adicione um identifi cador curto de localização como o nome de uma cidade ou ID do tile.
* **Preserve os dados brutos**: Nunca modifi que os downloads originais. Crie cópias processadas em um diretório separado.
* **Use CRS consistente**: Reprojete todos os datasets para um CRS comum no início do seu fl uxo de trabalho para evitar desalinhamento posterior.


Essas práticas podem parecer menores no início, mas se pagam rapidamente quando você precisa reproduzir resultados, compartilhar dados com colaboradores ou escalar para áreas de estudo maiores.